In [1]:
#IMPORTS
from sources.config import *

# PatientAggregator

In [2]:
from collections import defaultdict

class PatientAggregator:
    def __init__(self, strategy="mean", k=3, decision="weighted", thresholds=None):
        """
        strategy: 'mean' (baseline) ou 'topk' (melhorado)
        decision: 'weighted' (score * [5,3,1]) ou 'unweighted' (argmax puro)
        thresholds: tuplo opcional (th_present, th_unknown)
        """
        self.strategy: str= strategy
        self.k:int= k
        self.decision: str= decision
        self.thresholds = thresholds

    def decide(self, pr:np.ndarray, thresholds=None):
        th= thresholds or self.thresholds
        if th is not None:
            pass
        if self.decision=="weighted":
            return (pr * METRIC_W)
        else:
            return pr.argmax(1)
    

    def aggregate(self, probs, y_win, pids):
            
        """Group windows by patient"""
        groups = defaultdict(list) # diferente de {}
        labels = {}
        for p, y, pid in zip(probs, y_win, pids):
            #Se person_id==NaN, removemos a entrada
            if pd.isna(pid): continue
            pid_int = int(pid)
            groups[pid_int].append(p)
            labels[pid_int] = y

        ids = list(groups) #Cria uma lista de índices únicos. A LISTA NÃO ESTÁ ORDENA POR INT.

        """
        Calcula o vetor de probabilidades combinadas por classe [Present, Unknown, Absent]
        Combinação das previsões dos modelos ao longo das diferentes windows do paciente.

        """
        
        patient_probs = []
        for pid in ids:

            wins = np.array(groups[pid])
            n_wins = len(wins)

            if self.strategy == "mean":
                # Baseline original: média aritmética simples
                p_agg = np.mean(wins, axis=0)
            
            elif self.strategy == "topk":
                
                # Versão robusta: Top-k janelas para Present e Unknown
                eff_k = min(self.k, n_wins)
                top_pres = np.mean(wins[np.argsort(wins[:, 0])[-eff_k:], 0])
                top_unk = np.mean(wins[np.argsort(wins[:, 1])[-eff_k:], 1])
                p_abs = np.mean(wins[:, 2])

                total = top_pres + top_unk + p_abs
                p_agg = np.array([top_pres, top_unk, p_abs]) / total
            
            else:
                raise ValueError(f"Estratégia não implementada: {self.strategy}")

            #print(pid, p_agg)
            patient_probs.append(p_agg)

        """
        Calcula a probabilidade final do paciente para aquela sequência.
        Thresholds: (th_present), (th_unknown)
            (th_present) p[0]: Probabilidade final mínima para classificar como "Sopro cardíaco presente" (0). 
            (th_unknown) p[1]: Probabilidae final m~´inima para classificar como "Sopro cardíaco não foi encontrado" (1)
            - Se o tuplo for fornecido
            - Se o tuplo não fornecido
                - Adiciona viés às classes se atributo self.decision da classe Aggregator for "weighted"
                - A probabilidade será o maior valor registado.    
        
        y_pred = np.array(preds) é a coleção de valores previstos.
        """

        pr = np.array(patient_probs)
        y_true = np.array([labels[i] for i in ids])

        
        # Decisão final
        if self.thresholds is not None:
            th_pres, th_unk = self.thresholds
            preds = [0 if p[0] >= th_pres else (1 if p[1] >= th_unk else 2) for p in pr]
            y_pred = np.array(preds)
        
        else:
            score = pr * METRIC_W if self.decision == "weighted" else pr
            y_pred = score.argmax(axis=1)

        return y_true, y_pred, pr

#== Patient Aggregator ==#
agg = PatientAggregator(strategy="mean",k=3, decision="weighted", thresholds=None)

    

# Treinar modelo criado na deteção de sopros cardíacos

In [3]:
class PatientWACallback(keras.callbacks.Callback):
        
    def __init__(self, X_val, y_val, pids_val, aggregator):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.pids_val = pids_val
        self.aggregator = aggregator

    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.X_val, batch_size=128)
        yt, yp, _ = self.aggregator.aggregate(probs, self.y_val, self.pids_val)
        w = METRIC_W[yt]
        wa = float(np.sum(w * (yt == yp)) / np.sum(w))
        if isinstance(logs, dict):
            logs["val_wa"] = wa
        print(f" - val_wa (paciente): {wa:.4f}")


class MurmurTrainer:
    #Global class variable assignment

    def __init__(self, model:keras.models.Sequential, name, weight_mode="smoothed"):
        """weight_mode: 'standard' (inverso clássico), 'smoothed' (raiz quadrada) ou None"""
        self.model = model
        self.name = name
        self.weight_mode = weight_mode
        self.CFG=Config()
        self.seed=42

    def compute_weights(self, y_train):
        if self.weight_mode is None:
            return None
        counts = np.bincount(y_train, minlength=3).astype(float)
        if self.weight_mode == "standard":
            w = counts.sum() / (3.0 * counts)
        elif self.weight_mode == "smoothed":
            w = np.sqrt(counts.sum() / (3.0 * counts))
            w = w / w.mean()
        return {i: float(w[i]) for i in range(3)}


    def train(self, Xtr, ytr, Xva, yva, pva, aggregator:PatientAggregator, 
        metrics:list[str]=[],
        optimizer: dict[str, keras.optimizers.Optimizer]={"adam":keras.optimizers.Adam(learning_rate=1e-3)}, 
        loss: dict[str, keras.losses.Loss]={"sparse_categorical_crossentropy":keras.losses.SparseCategoricalCrossentropy()}, 
        epochs:int=0, batch_size:int=0):
        
        if any(self.model is m for m in MODELS.values()):
            print(
                f"WARNING: The current {self.model} object is being trained again"\
                "training will continue from the model's current weights."
            )
        
        optimizer_func= next(iter(optimizer.keys()))
        loss_func= next(iter(loss.keys()))
        metrics_list= metrics if (len(metrics)!=0) else ["accuracy"]
        
        self.model.compile(
            optimizer=next(iter(optimizer.values())),
            loss=next(iter(loss.values())),
            metrics=metrics_list
        )

        callbacks = [
            PatientWACallback(Xva, yva, pva, aggregator),
            keras.callbacks.EarlyStopping(monitor="val_wa", mode="max", patience=8, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1)
        ]

        t0 = time.time()

        history=self.model.fit(
            Xtr, ytr,
            validation_data=(Xva, yva),
            epochs=self.CFG.epochs if (epochs==0) else epochs,
            batch_size=self.CFG.batch_size if (batch_size==0) else batch_size,
            class_weight=self.compute_weights(ytr),
            callbacks=callbacks,
            verbose=2
        )
        t1 = time.time()-t0

        #== SAVE TRAINING METRICS ==#
        h = history.history

        HISTORIES[self.name] = dict(
            seed=self.seed,
            params=self.model.count_params(),
            weight_mode=self.weight_mode,
            loss=loss_func,
            optimizer=optimizer_func  
        )
        MODELS[self.name] = self.model
        RESULTS[self.name] = dict(
            params=self.model.count_params(),
            epochs=len(h["loss"]),
            train_acc=h["accuracy"][-1],
            val_acc=h["val_accuracy"][-1],
            best_val_acc=max(h["val_accuracy"]),
            val_loss=h["val_loss"][-1],
            min_val_loss=min(h["val_loss"]),
            gap=h["accuracy"][-1] - h["val_accuracy"][-1],
            fit_runtime=round(t1, 1),
        )

    
    def find_optimal_thresholds(self, probs_val, y_val, pids_val, aggregator):
        """Encontra o par (th_present, th_unknown) que maximiza o WA na validação."""
        yt, _, pr = aggregator.aggregate(probs_val, y_val, pids_val)
        
        best_wa = -1.0
        best_th = (0.33, 0.33)
        
        for th_pres in np.linspace(0.15, 0.75, 61):
            for th_unk in np.linspace(0.15, 0.65, 51):
                preds = np.array([
                    0 if p[0] >= th_pres else (1 if p[1] >= th_unk else 2)
                    for p in pr
                ])
                w = METRIC_W[yt]
                wa = float(np.sum(w * (yt == preds)) / np.sum(w))
                if wa > best_wa:
                    best_wa = wa
                    best_th = (round(float(th_pres), 3), round(float(th_unk), 3))
                    
        return best_th, best_wa

    """
    Create confusion matrix of model 

    """


    def evaluate_and_report(self, X, y, pids, aggregator, split_name="VALIDAÇÃO", optimize_thresholds=False):
            """Avalia o modelo ao nível do paciente, imprime WA e matriz de confusão."""
            probs = self.model.predict(X, batch_size=256)
            yt, yp, _ = aggregator.aggregate(probs, y, pids)
            
            w = METRIC_W[yt]
            wa = float(np.sum(w * (yt == yp)) / np.sum(w))
            
            print(f"\n{'='*45}")
            print(f" RESULTADOS: {split_name} | Estratégia: {aggregator.strategy.upper()}")
            print(f"{'='*45}")
            print(f"WA_murmur: {wa:.4f}")
            print(self._confusion_df(yt, yp))
            
            best_th = aggregator.thresholds
            if optimize_thresholds:
                best_th, best_wa = self.find_optimal_thresholds(probs, y, pids, aggregator)
                # Atualiza o agregador com os melhores limiares encontrados
                aggregator.thresholds = best_th
                _, yp_opt, _ = aggregator.aggregate(probs, y, pids)
                
                print(f"\n--- APÓS CALIBRAÇÃO DE LIMIARES {best_th} ---")
                print(f"WA_murmur Otimizado: {best_wa:.4f}")
                print(self._confusion_df(yt, yp_opt))
                
            return wa, best_th
    
    @staticmethod
    def _confusion_df(y_true, y_pred):
        m = np.zeros((3, 3), dtype=int)
        for t, p in zip(y_true, y_pred):
            m[p, t] += 1
        return pd.DataFrame(
            m,
            index=[f"Pred {c}" for c in CLASSES],
            columns=[f"True {c}" for c in CLASSES]
        )

In [4]:
#Iniciar o folder de resultados para esta run
#RUN_FOLDER=export_results(RESULTS=str(os.getenv('RESULTS')))

#LOAD current environment params:
RESULTS:dict[str, dict[str, int | float]] = {}     # experiment name -> summary dict
HISTORIES:dict[str, dict[str, list[float]]] = {}   # experiment name -> keras History.history dict
MODELS:dict[str, keras.Model] = {}      # experiment name -> trained model

#Load config variables: specific to this set of runs
CFG=Config()

import numpy as np
from pathlib import Path
import pandas as pd


In [5]:

"""
DATA LOADING AND KFOLD CROSS VALIDATION

"""

def load_split(architecture, name, CACHE):
    d = np.load(CACHE / f"{name}.npz")
    info = pd.read_csv(CACHE / f"{name}_info.csv", dtype={"pid": str})
    #Extrai do zip .npz o numpy array "cnn2d"
    return d[f"{architecture}"], d["y"], info["pid"].values

(Xtr), ytr, ptr = load_split(architecture="rnn", name="train", CACHE=Path(CACHE))
Xva, yva, pva = load_split(architecture="rnn", name="val", CACHE=Path(CACHE))

Xtr, ytr, ptr = (np.asarray(x) for x in (Xtr, ytr, ptr))
Xva, yva, pva = (np.asarray(x) for x in (Xva, yva, pva))

# Concatenar dados de treino e validação
X:np.ndarray=np.concatenate([Xtr, Xva])
y:np.ndarray=np.concatenate([ytr, yva])
pid:np.ndarray=np.concatenate([ptr, pva])

#Outer fold: Data splitter. Object will split fed data into K sets, grouped by patient id
outer_split = StratifiedGroupKFold(
    n_splits=CFG.K,
    shuffle=True,
    random_state=0
)

dummy_X = np.zeros(len(y)) #Quantas samples (casos registados) existem.

#Gera K folds ((train_idx, test_idx)), que são tuplos de np.Array, com dados de features(X), target class(y) e código do paciente(pid)
#Split apenas das posições (indexes) do dummy array, 
train_idx, test_idx = next(outer_split.split(dummy_X, y, pid))

#Garantir que não há data leakage de identidades: um paciente está presente na validação e no treino.
assert not set(pid[train_idx]) & set(pid[test_idx]), "Data leakage: doentes repetidos entre treino e fold"

#Inner splitter object
inner_split=StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=1)
#Split indexes of only training array
dummy_X=np.zeros(len(train_idx))

#Posições dentro do array de treino (train_idx)
#inner_split.split --> objeto Iterator gera folds step by step. 
# next() --> Guarda a primeira iteração do Iterator
#Evita construir "n_splits" e guardar em memória
#Gera posições relativas dentro de train_idx para o set de treino

a, b = next(inner_split.split(dummy_X, y[train_idx], pid[train_idx]))

#Recolher índices dos registos (linhas do array) para 2 efeitos:
    #FIT: rows used for weigth training
    #IN: rows used for val_wa/epoch
fit_idx, in_idx = train_idx[a], train_idx[b]



In [6]:

def base_model_build(model:models.Sequential, name:str, dropout_rate:float, activation:str, weight_mode:str):
    
    for i in CFG.epochs:

        tf.keras.utils.set_random_seed(CFG.seed)
        trainer=MurmurTrainer(
            model=model,
            name=name,
            weight_mode=weight_mode,
        )

        trainer.train(
            Xtr=X[fit_idx].astype("float32"),
            ytr=y[fit_idx],
            Xva=X[in_idx].astype("float32"),
            yva=y[in_idx],
            pva=pid[in_idx],
            aggregator=agg,
            epochs=i
        )

    import csv
    with open(FULL_RESULTS_TABLE_PATH, newline="", mode="w") as file:
        metrics = list(dict.fromkeys(k for inner in RESULTS.values() for k in inner))
        writer= csv.DictWriter(file, fieldnames=["model_name", *metrics])
        writer.writeheader()
        for model_name, cur_line in RESULTS.items():
            writer.writerow({"model_name":model_name, **cur_line})

    with open(PARAMS_TABLE_PATH, newline="", mode="w") as file:
        params = list(dict.fromkeys(k for inner in HISTORIES.values() for k in inner))
        writer= csv.DictWriter(file, fieldnames=["model_name", *params])
        writer.writeheader()
        for model_name, cur_line in HISTORIES.items():
            writer.writerow({"model_name":model_name, **cur_line})


    #==CALL MODEL TRAIN==#
    print("H>>> ",HISTORIES.items())
    print("R>>> ",len(RESULTS))
    print("M>>> ",len(MODELS))

    #== PRINT CONFUSION MATRIX ==#




In [7]:
"""
Calling designed ANN.

"""

from sources import models

base_model_build(
    model=models.McDonald_model,
    name="McDonald_Model",
    dropout_rate=0.1,
    activation='tanh',
    weight_mode="smoothed",
)
    


Epoch 1/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step
 - val_wa (paciente): 0.5072
106/106 - 78s - 735ms/step - accuracy: 0.7442 - loss: 0.5591 - val_accuracy: 0.7631 - val_loss: 0.7202 - val_wa: 0.5072 - learning_rate: 0.0010
Epoch 2/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step
 - val_wa (paciente): 0.6763
106/106 - 51s - 481ms/step - accuracy: 0.7662 - loss: 0.5006 - val_accuracy: 0.8192 - val_loss: 0.5776 - val_wa: 0.6763 - learning_rate: 0.0010
Epoch 3/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step
 - val_wa (paciente): 0.7101
106/106 - 55s - 522ms/step - accuracy: 0.8142 - loss: 0.4168 - val_accuracy: 0.7952 - val_loss: 0.5854 - val_wa: 0.7101 - learning_rate: 0.0010
H>>>  dict_items([('McDonald_Model', {'seed': 42, 'params': 199615, 'weight_mode': 'smoothed', 'loss': 'sparse_categorical_crossentropy', 'optimizer': 'adam'})])
R>>>  1
M>>>  1


In [8]:
def weighted_accuracy(y_true:np.ndarray, y_pred:np.ndarray):
    w = METRIC_W[y_true]
    return float(np.sum(w*(y_true==y_pred)) / np.sum(w))

In [9]:
"""
Export predictions to ./results
Evaluate model's WA
Render accuracy and loss for the best model in RESULTADOS 

"""

from sklearn.metrics import confusion_matrix

def evaluate(model:keras.Sequential, X:np.ndarray, y:np.ndarray, pids:np.ndarray, agg:PatientAggregator, name:str, thresholds=None):
    probs=model.predict(x=X, batch_size=CFG.batch_size)
    y_true, _, pr = agg.aggregate(probs, y, pids)
    y_predicted=agg.decide(pr=pr, thresholds=thresholds)
    
    print(f"{model.name}: WA = {weighted_accuracy(y_true=y_true, y_pred=y_predicted):.4f}")
    print(confusion_matrix(y_true=y_true, y_pred=y_predicted))
    
    return y_true, y_predicted